# Eval Metrics Part → Second Leg of Hypothesis Tests (**Metrics Engine**)

In **Leg 2**, we stop “creating signals” and start **measuring performance**.

The key idea is simple:

> If Leg 1 tells us **where** a hypothesis applies (events) and **what it predicts**,  
> Leg 2 tells us **how well it actually works**.

- This notebook exists because we want **one consistent evaluation system** for *all* hypotheses, instead of writing different metric code every time. 
- Given an event definition and (optional) predictions, compute **all evaluation metrics** in a standardized way.

You can think of `eval_metrics.ipynb` as a reusable **metrics calculator**.


## The 3-Leg Pipeline (How the whole system is organized)

### **Leg 1 — prediction.ipynb (Event Factory)** 
**Goal:** Create new columns that mark **where** each hypothesis applies and **what the rule predicts** (if applicable).

- We do **NOT** prove anything here.
- We do **NOT** compute p-values or draw conclusions here.
- We only build:
  - **event masks** (which rows are “in the hypothesis world”)
  - **baseline rule predictions** (optional for some hypotheses)

**Output pattern (typical):**
- `is_Hx` → 0/1 mask for “this row is a valid event for Hx”
- `pred_Hx_*` → rule-based prediction (if the hypothesis is directional)
- additional tags (if the hypothesis is grouping / moderator)

Some hypotheses are already fully defined by existing features and labels → they may pass through this leg with minimal work.

### **Leg 2 — eval_metrics.ipynb (Metrics Engine)** --> WE ARE IN HERE !!!
**Goal:** Given an event definition and predictions, compute all **evaluation metrics** consistently.

Think of this as a reusable “calculator”:

- Input: **(mask, y_true, y_pred)**  
- Output: metrics such as:
  - **hit-rate**
  - **precision / recall / F1** (for rule predictions)
  - **p-value** (example: test `hit-rate > 0.5`)
  - **effect size** (example: signed returns, distance reduction)
  - reusable slice logic (IB width, gap alignment, etc.)

### **Leg 3 — hypothesis_tests.ipynb (Report + Decision Layer)**
**Goal:** Present results in a clean, viewer-friendly form and state decisions clearly.

This is where we:
- show tables/figures
- explain measurement choices (especially for “reversion” type ideas)
- decide:
  - **Reject null hypothesis** or
  - **Fail to reject null hypothesis**
based on the metrics and p-values from Leg 2.


## What we feed into the Metrics Engine → Inputs

For any hypothesis (H1–H5), we only need a small set of objects:

- **`mask`**  
  A boolean or 0/1 filter that selects the rows we are evaluating  
  (example: `is_H4_nowhip == 1`)

- **`y_true`**  
  The true future outcome we want to evaluate against  
  (example: `dir15`, `dir30`, `ret15`, `ret30`, or a custom target like distance-to-mid reduction)

- **`y_pred`** *(only if the hypothesis produces a rule prediction)*  
  The rule-based predicted outcome  
  (example: `pred_H2_dir15`, `pred_H4_dir30`)

So the core interface is:

- **Input:** **(mask, y_true, y_pred)**  
- **Output:** a consistent set of results and statistics


## What the Metrics Engine returns → Outputs

Depending on the hypothesis type, `eval_metrics.ipynb` can compute metrics like:

- **Hit-rate (accuracy)**  
  “On the masked event rows, what fraction of predictions were correct?”

- **Precision / Recall / F1** *(for direction/rule hypotheses)*  
  Extra classification quality metrics, especially useful when class balance matters.

- **p-value (statistical significance)**  
  Examples:
  - Test whether **hit-rate > 0.5** (better than random guessing)
  - Test whether **mean return ≠ 0** (or > 0 / < 0 depending on hypothesis)

- **Effect size (economic/statistical magnitude)**  
  Examples:
  - mean/median **signed returns** (`ret15`, `ret30`)
  - “distance reduction” style outcomes for reversion hypotheses  
    (example: did `|close_f15 - ib_mid|` shrink vs now?)

- **Reusable “slice” logic (conditional analysis)**  
  The same hypothesis can be evaluated under conditions such as:
  - **IB width regime:** narrow vs wide (`is_H6_narrow`, `is_H6_wide`)
  - **Gap alignment:** aligned vs not (`is_H7_align`)
  - other sensitivity filters (like whipsaw vs no-whipsaw)

    * This allows clean comparisons like:
      - “Does H2 work better on wide-IB days?”
      - “Does H4 improve when whipsaw is excluded?”
      - “Are returns larger when gap alignment is present?”

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path